#  Griffin Library - Intelligent Book Recommendation System

## Notebook 04 · Feature Engineering

**Goal:** Transform the clean merged dataset into a model-ready feature set.  
Build the weighted rating score, normalize signals, clean genres,  
and construct the final embedding text for FAISS indexing.

| | Details |
|---|---|
| **Input** | `data/processed/books_merged.csv` |
| **Operations** | Bayesian smoothing · Log normalization · Genre cleaning · Embedding text finalization |
| **Output** | `data/processed/books_features.csv` |
| **Next Step** | `05_model.ipynb` - Train similarity index and build recommendation engine |

---

In [1]:
import sys
# Dependencies installed via requirements.txt

import pandas as pd
import numpy as np
import os

os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

df = pd.read_csv("data/processed/books_merged.csv")

print(f"Loaded : {df.shape}")
print(f"Columns: {list(df.columns)}")

Loaded : (8577, 12)
Columns: ['book_id', 'title', 'authors', 'genres', 'avg_rating', 'num_ratings', 'description', 'summary', 'embedding_text', 'tier', 'url', 'num_pages']


In [2]:
# ═══════════════════════════════════════════════════════════════
# STEP 1 - Bayesian Weighted Rating Score
# ═══════════════════════════════════════════════════════════════
# Formula: weighted_rating = (v / (v + m)) * R + (m / (v + m)) * C
# Where:
#   R = book's average rating
#   v = number of ratings for the book
#   m = minimum ratings threshold (25th percentile)
#   C = mean rating across all books

C = df["avg_rating"].mean()
m = df["num_ratings"].quantile(0.25)

df["weighted_rating"] = (
    (df["num_ratings"] / (df["num_ratings"] + m)) * df["avg_rating"] +
    (m / (df["num_ratings"] + m)) * C
)

print(f"Global mean rating (C) : {C:.4f}")
print(f"Min ratings threshold  : {m:.0f}")
print(f"\nWeighted Rating Distribution:")
print(df["weighted_rating"].describe().round(4))

print(f"\nTop 10 books by weighted rating:")
print(df.nlargest(10, "weighted_rating")[["title","authors","avg_rating","num_ratings","weighted_rating"]])

Global mean rating (C) : 4.0379
Min ratings threshold  : 4041

Weighted Rating Distribution:
count    8577.0000
mean        4.0333
std         0.1813
min         2.6180
25%         3.9369
50%         4.0407
75%         4.1359
max         4.7504
Name: weighted_rating, dtype: float64

Top 10 books by weighted rating:
                                                  title            authors  \
4629     Words of Radiance (The Stormlight Archive, #2)  Brandon Sanderson   
1019                     The Complete Calvin and Hobbes     Bill Watterson   
3651                                       Know My Name      Chanel Miller   
719       The Way of Kings (The Stormlight Archive, #1)  Brandon Sanderson   
3550               Kingdom of Ash (Throne of Glass, #7)      Sarah J. Maas   
2563  A Court of Mist and Fury (A Court of Thorns an...      Sarah J. Maas   
223   The Essential Calvin and Hobbes: A Calvin and ...     Bill Watterson   
15    Harry Potter and the Deathly Hallows (Harry Po...    

### Bayesian Weighted Rating Score

- **Formula**: `weighted_rating = (v/(v+m)) * R + (m/(v+m)) * C`
- **C (global mean)**: 4.038 - the prior that pulls uncertain ratings toward
- **m (threshold)**: 4,041 ratings - books below this are pulled toward the global mean
- **Effect**: books with few ratings are penalized, popular well-rated books rise to the top
- **Result**: max weighted rating is 4.75 vs raw max of 5.0 - no perfect scores, realistic ranking
- **Top books make sense**: Brandon Sanderson, J.K. Rowling, Sarah J. Maas - genuinely beloved authors

---

In [3]:
# ═══════════════════════════════════════════════════════════════
# Normalize Signals
# ═══════════════════════════════════════════════════════════════

# Log-transform num_ratings (highly skewed)
df["log_ratings"] = np.log1p(df["num_ratings"])

# Normalize weighted_rating to [0, 1]
wr_min = df["weighted_rating"].min()
wr_max = df["weighted_rating"].max()
df["rating_norm"] = (df["weighted_rating"] - wr_min) / (wr_max - wr_min)

# Normalize log_ratings to [0, 1]
lr_min = df["log_ratings"].min()
lr_max = df["log_ratings"].max()
df["popularity_norm"] = (df["log_ratings"] - lr_min) / (lr_max - lr_min)

print("=== rating_norm ===")
print(df["rating_norm"].describe().round(4))

print("\n=== popularity_norm ===")
print(df["popularity_norm"].describe().round(4))

print("\nSample:")
print(df[["title","weighted_rating","rating_norm","num_ratings","popularity_norm"]].head(5))

=== rating_norm ===
count    8577.0000
mean        0.6637
std         0.0850
min         0.0000
25%         0.6185
50%         0.6672
75%         0.7118
max         1.0000
Name: rating_norm, dtype: float64

=== popularity_norm ===
count    8577.0000
mean        0.4652
std         0.2000
min         0.0000
25%         0.3610
50%         0.5049
75%         0.6032
max         1.0000
Name: popularity_norm, dtype: float64

Sample:
                                               title  weighted_rating  \
0                              To Kill a Mockingbird         4.269835   
1  Harry Potter and the Philosopher’s Stone (Harr...         4.469812   
2                                Pride and Prejudice         4.279752   
3                          The Diary of a Young Girl         4.179836   
4                                        Animal Farm         3.980065   

   rating_norm  num_ratings  popularity_norm  
0     0.774632      5691311         0.959648  
1     0.868414      9278135         1

### Normalize Signals

- **Log-transform num_ratings**: reduces the effect of extreme popularity (9.2M ratings for HP) - prevents one book from dominating
- **rating_norm [0,1]**: mean 0.664, std 0.085 - well-centered, ready for hybrid scoring
- **popularity_norm [0,1]**: mean 0.465, std 0.200 - wider spread, good discriminative signal
- **Harry Potter #1 gets popularity_norm = 1.0**: correct - most rated book in catalog
- **Both signals are independent**: correlation between them is low (0.058 from EDA) - complementary features

---

In [4]:
# ═══════════════════════════════════════════════════════════════
# Clean Genres
# ═══════════════════════════════════════════════════════════════

# Noise genres to remove
NOISE_GENRES = {
    "audiobook", "novels", "historical", "owned",
    "to-read", "currently-reading", "default",
    "books", "library", "favourites", "favorite"
}

def clean_genres(genre_str):
    if pd.isna(genre_str) or genre_str == "":
        return ""
    genres = [g.strip() for g in genre_str.split(",")]
    cleaned = [g for g in genres if g.lower() not in NOISE_GENRES and len(g) > 1]
    return ", ".join(cleaned)

df["genres_clean"] = df["genres"].apply(clean_genres)

# Stats
before = df["genres"].str.split(",").apply(lambda x: len(x) if isinstance(x, list) else 0).mean()
after  = df["genres_clean"].str.split(",").apply(lambda x: len(x) if isinstance(x, list) else 0).mean()

print(f"Avg genres before cleaning : {before:.2f}")
print(f"Avg genres after cleaning  : {after:.2f}")
print(f"\nSample cleaned genres:")
for g in df["genres_clean"].head(5):
    print(f"  • {g}")

Avg genres before cleaning : 6.55
Avg genres after cleaning  : 6.18

Sample cleaned genres:
  • Classics, Fiction, Historical Fiction, School, Literature, Young Adult
  • Fantasy, Fiction, Young Adult, Magic, Childrens, Middle Grade, Classics
  • Classics, Fiction, Romance, Historical Fiction, Literature
  • Classics, Nonfiction, History, Biography, Memoir, Holocaust
  • Classics, Fiction, Dystopia, Fantasy, Politics, School, Literature


### Clean Genres

- Removed noise tags: `audiobook`, `novels`, `historical`, `owned`, `to-read`, etc.
- Average genres per book reduced from 6.55 to 6.18 - minor cleanup, signal preserved
- All remaining genres are meaningful content descriptors ready for filtering and embedding

---

In [5]:
# ═══════════════════════════════════════════════════════════════
# Rebuild Final Embedding Text
# ═══════════════════════════════════════════════════════════════

def build_final_embedding(row):
    parts = []

    # Title + Author
    parts.append(f"{row['title']} by {row['authors']}")

    # Clean genres
    if row["genres_clean"]:
        parts.append(row["genres_clean"])

    # Goodreads description
    desc = str(row["description"]).strip()
    if desc and desc != "nan":
        parts.append(desc[:800])

    # CMU summary (Tier 1 only)
    if pd.notna(row["summary"]):
        summary = str(row["summary"]).strip()
        if summary:
            parts.append(summary[:800])

    return " | ".join(parts)

df["embedding_text_final"] = df.apply(build_final_embedding, axis=1)

avg_len = df["embedding_text_final"].str.len().mean()
min_len = df["embedding_text_final"].str.len().min()
max_len = df["embedding_text_final"].str.len().max()

print(f"Embedding text rebuilt for {len(df):,} books")
print(f"Avg length : {avg_len:.0f} chars")
print(f"Min length : {min_len} chars")
print(f"Max length : {max_len} chars")

print(f"\nTier 1 sample:")
print(df[df['tier']==1]['embedding_text_final'].iloc[0][:400])
print(f"\nTier 2 sample:")
print(df[df['tier']==2]['embedding_text_final'].iloc[0][:400])

Embedding text rebuilt for 8,577 books
Avg length : 946 chars
Min length : 43 chars
Max length : 1801 chars

Tier 1 sample:
To Kill a Mockingbird by Harper Lee | Classics, Fiction, Historical Fiction, School, Literature, Young Adult | The unforgettable novel of a childhood in a sleepy Southern town and the crisis of conscience that rocked it. "To Kill A Mockingbird" became both an instant bestseller and a critical success when it was first published in 1960. It went on to win the Pulitzer Prize in 1961 and was later ma

Tier 2 sample:
Harry Potter and the Philosopher’s Stone (Harry Potter, #1) by J.K. Rowling | Fantasy, Fiction, Young Adult, Magic, Childrens, Middle Grade, Classics | Harry Potter thinks he is an ordinary boy - until he is rescued by an owl, taken to Hogwarts School of Witchcraft and Wizardry, learns to play Quidditch and does battle in a deadly duel. The Reason ... HARRY POTTER IS A WIZARD!


### Rebuild Final Embedding Text

- Rebuilt embedding text using cleaned genres instead of raw genres
- Capped description at 800 chars and CMU summary at 800 chars - balanced contribution from both sources
- **Average length: 946 chars** - optimal range for sentence-transformers (all-mpnet-base-v2)
- **Min length: 43 chars** - a few very short entries, acceptable edge cases
- **Max length: 1,801 chars** - well within model token limits (~512 tokens ≈ ~2,000 chars)

---

In [6]:
# ═══════════════════════════════════════════════════════════════
# Select Final Features & Save
# ═══════════════════════════════════════════════════════════════

df_final = df[[
    "book_id", "title", "authors", "genres_clean",
    "avg_rating", "weighted_rating", "rating_norm",
    "num_ratings", "popularity_norm",
    "description", "summary", "embedding_text_final",
    "tier", "url",
    "num_pages",
]].rename(columns={
    "genres_clean"        : "genres",
    "embedding_text_final": "embedding_text",
})

# Ensure num_pages is clean integer (0 if missing)
df_final["num_pages"] = df_final["num_pages"].fillna(0).astype(int)

df_final.to_csv("data/processed/books_features.csv", index=False)

print(f"Saved: data/processed/books_features.csv")
print(f"\nFinal feature set:")
print(f"  Books           : {len(df_final):,}")
print(f"  Columns         : {len(df_final.columns)}")
print(f"  Columns         : {list(df_final.columns)}")
print(f"  With page count : {(df_final['num_pages'] > 0).sum():,} / {len(df_final):,}")
print(f"\nSample row:")
print(df_final.iloc[0][["title","genres","weighted_rating","rating_norm","popularity_norm","tier","num_pages"]])


Saved: data/processed/books_features.csv

Final feature set:
  Books           : 8,577
  Columns         : 15
  Columns         : ['book_id', 'title', 'authors', 'genres', 'avg_rating', 'weighted_rating', 'rating_norm', 'num_ratings', 'popularity_norm', 'description', 'summary', 'embedding_text', 'tier', 'url', 'num_pages']
  With page count : 1,084 / 8,577

Sample row:
title                                          To Kill a Mockingbird
genres             Classics, Fiction, Historical Fiction, School,...
weighted_rating                                             4.269835
rating_norm                                                 0.774632
popularity_norm                                             0.959648
tier                                                               1
num_pages                                                        323
Name: 0, dtype: object


---

## ✅ Feature Engineering Summary

| Step | Operation | Result |
|---|---|---|
| 1 | Bayesian weighted rating | Reliable score - penalizes books with few ratings |
| 2 | Log-normalize signals | `rating_norm` and `popularity_norm` in [0,1] |
| 3 | Clean genres | Removed noise tags - 6.18 avg genres/book |
| 4 | Rebuild embedding text | 946 avg chars - optimal for sentence-transformers |
| 5 | Save final feature set | `data/processed/books_features.csv` |


| Feature | Description | Used For |
|---|---|---|
| `weighted_rating` | Bayesian smoothed rating | Hybrid ranking |
| `rating_norm` | Normalized weighted rating [0,1] | FAISS hybrid score |
| `popularity_norm` | Log-normalized num_ratings [0,1] | FAISS hybrid score |
| `genres` | Cleaned multi-label genres | Filtering + embedding |
| `embedding_text` | title + genres + description + summary | FAISS semantic search |
| `tier` | 1 = with CMU, 2 = description only | Quality weighting |

 **Next → `05_model.ipynb`**